# Fast Isomorphism

## Prompt

The magma method 'isomorphic' in my Python module 'https://github.com/alreich/finalg' is a brute force method of determining whether two algebras are isomorphic. It can only be used for very small order finite algebras. I'd like to come up with a method of making it more efficient, so that it works for larger algebras. I think that by first looking for isomorphisms between subalgebras (subgroups) of the two algebras being compared, it might be made more efficient. I started working on that idea in 'scratch/work_in_progress.py' but it is unfinished. Would you please try to write a method that can determine whether two algebras, as represented in 'finalg' with Cayley tables, are isomorphic by considering the properties of the algebras and whether their subalgebras are isomorphic. It doesn't have to work for very large algebras, but it should be faster than the existing 'isomporphic' method.

## Answer

Now let's look at the work-in-progress file and about_subalgebras.py.There's already an `element_order` method in Monoid. Let me check monoid.py and group.py more fully, plus finite_algebra_ABC.py.Good. Now let's check `finalg/__init__.py` for exported names, and `examples.py` for pre-built example algebras (potentially larger ones for testing).Now insert the new methods right after `isomorphic()`:Now let's install and test correctness against the existing brute-force `isomorphic` method across the test suite examples, plus timing comparisons.Now let's run the full existing test suite to make sure nothing broke, and do a broader randomized fuzz comparison + timing benchmark on larger groups.All 525 existing tests still pass (unchanged behavior). Now let's fuzz-test correctness against the brute-force method across many algebra types/sizes, and benchmark speed.Excellent — `fast_isomorphic` handles S4 (order 24, where brute force needs 24! ≈ 6×10²³ permutations) in 0.16s. Now let's also test non-isomorphic negative cases at larger sizes, and the Magma (non-associative) case to confirm the generic invariant logic still works there.`fast_isomorphic` handles this instantly where the original brute-force method times out. Let's test IP_Loop and mag_id too, and also verify negative (non-isomorphic) cases and A5 (order 60).Now let's test A5 (order 60) and a couple of non-isomorphic large-order negative cases, plus test against all pairs of `examples` of equal order for consistency with `isomorphic` where feasible.All correct, and dramatically faster for the negative cases too (invariant mismatch caught instantly). Let's also add proper unit tests to the test suite matching existing conventions, then run the whole suite once more.Good, this gives me the exact test style to match. Let me add a parallel `TestFastIsomorphism` class right after `TestIsomorphism`.All 533 tests pass. Now let's clean up the scratch work-in-progress file (mark it superseded) and check the git diff before committing.I built and tested this against your actual `finalg` repo. Here's what I did and how it works.

## The idea, made concrete

Your instinct about subalgebras was right, but there's a cleaner way to use it than finding one big subalgebra and matching remainder elements (which is what `work_in_progress.py` was doing): use a **generating set** instead. Any isomorphism is completely determined by where it sends a small set of generators, because everything else is forced by repeated products — that's exactly the "chain of subalgebras" idea, just applied directly to the search instead of as a separate subalgebra-matching step.

So `fast_isomorphic` (added to `Magma`, inherited by every subclass — Group, Ring, Quasigroup, etc.):

1. **Cheap invariant checks first** — class, order, identity, commutativity — same as `isomorphic` already does.
2. **Per-element invariants.** For each element `x`, it computes a "fingerprint": the sorted fiber-size profile of the row `y ↦ x*y` and column `y ↦ y*x` (how many `y` produce each result — provably preserved by any isomorphism, even for non-associative magmas), plus idempotency and element order where available. If the multisets of fingerprints don't match between the two algebras, they can't be isomorphic — instant `False`.
3. **A small generating set** for `self` (reusing your existing `generators()`).
4. Only tries images for *those* generators, restricted to elements in `other` with a matching fingerprint — then extends each candidate to a full map by propagating products (the same growth process as `closure()`, but building the mapping alongside it), bailing out the instant a conflict appears.

This shrinks the search from `other.order!` down to roughly `(candidates per generator)^(number of generators)`, with most bad candidates dying after a couple of multiplications instead of a full table check.

## Results against your test suite

All 525 existing tests still pass unchanged, plus 7 new tests I added mirroring your `TestIsomorphism` class. Benchmarks against the existing `isomorphic`:

| Algebra | order | `isomorphic` | `fast_isomorphic` |
|---|---|---|---|
| Z9 | 9 | 0.052s | 0.002s |
| Z10 | 10 | 1.03s | 0.002s |
| D4 vs Q8 vs Z8 (non-isomorphic) | 8 | ~0.68s each | <0.001s each |
| Latin_Sqr quasigroup (non-associative!) | 10 | **times out (>20s)** | instant |
| S4 | 24 | infeasible (24! ≈ 6×10²³) | 0.16s |
| A5 | 60 | infeasible | ~10s |

It also works correctly on non-associative Magmas and Quasigroups/Loops (no identity, no associativity needed anywhere in the algorithm).

The patch is attached (`fast_isomorphic.patch`, applies cleanly with `git apply` from your repo root), along with the full updated `magma.py` and `test_magma.py` for reference. One note: I left `scratch/work_in_progress.py` alone since you may still want it, but it's now superseded — feel free to delete it once you've reviewed this.